# Week 06 — Validation and Research Audit

## 1. Two Paper Findings and My Methodology Questions

This notebook audits both the research-paper findings and my own Week 5 K-Means model.

The purpose is not to criticize the research paper. The purpose is to practice checking whether methodology, validation, and claim strength support the conclusions.

### Finding 1: Higher-performing content showed stronger search visibility and traffic

The research found that one content-performance archetype showed higher observed impressions, clicks, pageviews, and organic sessions.

**My methodology question:**

How were the performance archetypes defined and validated independently of the same metrics used to describe them?

This question matters because if the clusters are created using impressions, clicks, pageviews, and organic sessions, then finding that one cluster has higher values on those same variables may partly be a direct consequence of the clustering method.

A stronger interpretation would clearly distinguish between the variables used to create the groups and any independent evidence used to validate their practical meaning.

### Finding 2: Two content-performance archetypes were identified

The research selected two clusters because K = 2 produced the strongest silhouette score among the tested cluster counts.

**My methodology question:**

Does the validation design show that the two-cluster structure remains stable on different time periods or independent groups of content?

A silhouette score measures how well-separated clusters are within the evaluated data, but it does not by itself prove that the same structure will remain stable when new data is observed.

A stronger validation design could compare the cluster structure across different time periods or independent groups.

In [ ]:
!pip install -q duckdb pandas numpy scikit-learn pyarrow

import duckdb
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
HF_TOKEN=

In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

dataset = "hf://datasets/FlyRank/internship-warehouse"

df_raw = con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{dataset}/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 500000
""").df()

print("Dataset shape:", df_raw.shape)

print("\nAll columns:")
for column in df_raw.columns:
    print(column)

df_raw.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (500000, 31)

All columns:
report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [ ]:
date_columns = []

for column in df_raw.columns:
    column_lower = column.lower()

    if "date" in column_lower or "time" in column_lower:
        date_columns.append(column)

print("Possible date/time columns:")
print(date_columns)

Possible date/time columns:
['report_date']


In [ ]:
print("Possible date/time columns:", date_columns)

for column in date_columns:
    print(f"\n--- {column} ---")
    print(df_raw[column].head(10))
    print("Data type:", df_raw[column].dtype)

Possible date/time columns: ['report_date']

--- report_date ---
0   2025-01-27
1   2025-01-27
2   2025-01-27
3   2025-01-27
4   2025-01-27
5   2025-01-27
6   2025-01-27
7   2025-01-27
8   2025-01-27
9   2025-01-27
Name: report_date, dtype: datetime64[us]
Data type: datetime64[us]


In [ ]:
print(df_raw.dtypes)

report_date                 datetime64[us]
client_hash_id                      object
content_hash_id                     object
client_has_gsc                        bool
client_has_ga4                        bool
gsc_data_available                    bool
ga4_data_available                    bool
gsc_impressions                      int64
gsc_clicks                           int64
gsc_sum_position                     Int64
gsc_avg_position                   float64
ga4_pageviews                        int64
ga4_sessions                         int64
ga4_users                            int64
ga4_engaged_sessions                 int64
ga4_total_engagement_sec             int64
sessions_organic                     int64
sessions_direct                      int64
sessions_referral                    int64
sessions_social                      int64
sessions_paid                        int64
sessions_ai                          int64
ai_chatgpt                           int64
ai_perplexi

In [ ]:
DATE_COLUMN = "report_date"

df_raw[DATE_COLUMN] = pd.to_datetime(df_raw[DATE_COLUMN])

df_raw = df_raw.sort_values(DATE_COLUMN).reset_index(drop=True)

print("Earliest date:", df_raw[DATE_COLUMN].min())
print("Latest date:", df_raw[DATE_COLUMN].max())

df_raw[[DATE_COLUMN]].head()

Earliest date: 2025-01-27 00:00:00
Latest date: 2025-04-30 00:00:00


,report_date
0,2025-01-27
1,2025-01-27
2,2025-01-27
3,2025-01-27
4,2025-01-27


In [ ]:
# Calculate the split point: first 80% of time for training,
# last 20% of time for honest evaluation

split_index = int(len(df_raw) * 0.80)

train_raw = df_raw.iloc[:split_index].copy()
test_raw = df_raw.iloc[split_index:].copy()

print("TIME-AWARE SPLIT")
print("\nTraining data:")
print("Rows:", len(train_raw))
print("Date range:",
      train_raw[DATE_COLUMN].min(),
      "to",
      train_raw[DATE_COLUMN].max())

print("\nEvaluation data:")
print("Rows:", len(test_raw))
print("Date range:",
      test_raw[DATE_COLUMN].min(),
      "to",
      test_raw[DATE_COLUMN].max())

TIME-AWARE SPLIT

Training data:
Rows: 400000
Date range: 2025-01-27 00:00:00 to 2025-04-19 00:00:00

Evaluation data:
Rows: 100000
Date range: 2025-04-19 00:00:00 to 2025-04-30 00:00:00


In [ ]:
# Aggregate each content item in the training period

train_content = train_raw.groupby("content_hash_id").agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    total_pageviews=("ga4_pageviews", "sum"),
    total_sessions=("ga4_sessions", "sum"),
    total_users=("ga4_users", "sum"),
    total_engaged_sessions=("ga4_engaged_sessions", "sum"),
    total_engagement_time=("ga4_total_engagement_sec", "sum"),
    organic_sessions=("sessions_organic", "sum"),
    direct_sessions=("sessions_direct", "sum"),
    referral_sessions=("sessions_referral", "sum"),
    social_sessions=("sessions_social", "sum"),
    ai_sessions=("sessions_ai", "sum"),
    total_scroll_events=("scroll_events", "sum")
).reset_index()


# Aggregate each content item in the later evaluation period

test_content = test_raw.groupby("content_hash_id").agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    total_pageviews=("ga4_pageviews", "sum"),
    total_sessions=("ga4_sessions", "sum"),
    total_users=("ga4_users", "sum"),
    total_engaged_sessions=("ga4_engaged_sessions", "sum"),
    total_engagement_time=("ga4_total_engagement_sec", "sum"),
    organic_sessions=("sessions_organic", "sum"),
    direct_sessions=("sessions_direct", "sum"),
    referral_sessions=("sessions_referral", "sum"),
    social_sessions=("sessions_social", "sum"),
    ai_sessions=("sessions_ai", "sum"),
    total_scroll_events=("scroll_events", "sum")
).reset_index()

print("Training content items:", train_content.shape)
print("Evaluation content items:", test_content.shape)

train_content.head()

Training content items: (13422, 15)
Evaluation content items: (12569, 15)


,content_hash_id,total_impressions,total_clicks,avg_position,total_pageviews,total_sessions,total_users,total_engaged_sessions,total_engagement_time,organic_sessions,direct_sessions,referral_sessions,social_sessions,ai_sessions,total_scroll_events
0,content_000005d4ced12088,64,1,30.180714,0,0,0,0,0,0,0,0,0,0,0
1,content_0002bd310bf01f15,1590,0,50.685151,0,0,0,0,0,0,0,0,0,0,0
2,content_00033c286cc93446,206,0,55.263509,0,0,0,0,0,0,0,0,0,0,0
3,content_000d3f2ab6f6e376,92,4,18.565092,0,0,0,0,0,0,0,0,0,0,0
4,content_0017cf2a6ec5895a,493,0,60.927996,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
features = [
    "total_impressions",
    "total_clicks",
    "avg_position",
    "total_pageviews",
    "total_sessions",
    "total_users",
    "total_engaged_sessions",
    "total_engagement_time",
    "organic_sessions",
    "direct_sessions",
    "referral_sessions",
    "social_sessions",
    "ai_sessions",
    "total_scroll_events"
]

# Clean training data
train_content[features] = train_content[features].copy()

train_content["avg_position"] = train_content["avg_position"].fillna(
    train_content["avg_position"].median()
)

train_content[features] = train_content[features].fillna(0)


# Clean evaluation data
# Use the TRAINING median to avoid using future evaluation information
test_content["avg_position"] = test_content["avg_position"].fillna(
    train_content["avg_position"].median()
)

test_content[features] = test_content[features].fillna(0)


print("Missing values in training data:")
print(train_content[features].isnull().sum().sum())

print("\nMissing values in evaluation data:")
print(test_content[features].isnull().sum().sum())

X_train = train_content[features].copy()
X_test = test_content[features].copy()

print("\nTraining shape:", X_train.shape)
print("Evaluation shape:", X_test.shape)

Missing values in training data:
0

Missing values in evaluation data:
0

Training shape: (13422, 14)
Evaluation shape: (12569, 14)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Fit scaler ONLY on earlier/training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

# Transform later/evaluation data using the same scaler
X_test_scaled = scaler.transform(X_test)

print("Training data scaled:", X_train_scaled.shape)
print("Evaluation data scaled:", X_test_scaled.shape)

Training data scaled: (13422, 14)
Evaluation data scaled: (12569, 14)


In [ ]:
# Test different cluster counts
time_results = []

for k in range(2, 7):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    # Fit ONLY on earlier/training data
    train_labels = kmeans.fit_predict(X_train_scaled)

    # Predict clusters for later/evaluation data
    test_labels = kmeans.predict(X_test_scaled)

    # Evaluate cluster separation
    train_score = silhouette_score(X_train_scaled, train_labels)
    test_score = silhouette_score(X_test_scaled, test_labels)

    time_results.append({
        "k": k,
        "train_silhouette": train_score,
        "later_time_silhouette": test_score
    })

time_results_df = pd.DataFrame(time_results)

time_results_df

,k,train_silhouette,later_time_silhouette
0,2,0.841583,0.880785
1,3,0.519433,0.594745
2,4,0.541614,0.600809
3,5,0.544295,0.596599
4,6,0.473428,0.552566


In [ ]:
# Replace this with your actual Week 5 held-out silhouette score
random_split_score = 0.0000

# Best result from the honest time-aware evaluation
best_time_row = time_results_df.loc[
    time_results_df["later_time_silhouette"].idxmax()
]

best_time_k = int(best_time_row["k"])
time_aware_score = float(best_time_row["later_time_silhouette"])

comparison_df = pd.DataFrame({
    "Validation Design": [
        "Week 5 Random 80/20 Split",
        "Week 6 Time-Aware Split"
    ],
    "Best K": [
        2,  # Replace if your Week 5 best K was different
        best_time_k
    ],
    "Evaluation Metric": [
        "Held-out silhouette score",
        "Later-time silhouette score"
    ],
    "Score": [
        random_split_score,
        time_aware_score
    ]
})

comparison_df

,Validation Design,Best K,Evaluation Metric,Score
0,Week 5 Random 80/20 Split,2,Held-out silhouette score,0.000000
1,Week 6 Time-Aware Split,2,Later-time silhouette score,0.880785


In [ ]:
# Create a leakage audit table

leakage_audit = pd.DataFrame({
    "Feature_or_Process": [
        "content_hash_id",
        "Search impressions",
        "Search clicks",
        "Average search position",
        "Pageviews and sessions",
        "Traffic-source sessions",
        "Engagement and scroll events",
        "Feature scaling",
        "Time-aware evaluation"
    ],

    "Leakage_Risk": [
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Controlled",
        "Controlled"
    ],

    "Audit_Result": [
        "Used only as an identifier and excluded from K-Means features",
        "Observed performance signal available in the relevant period",
        "Observed performance signal available in the relevant period",
        "Observed performance signal available in the relevant period",
        "Observed performance signals available in the relevant period",
        "Observed performance signals available in the relevant period",
        "Observed performance signals available in the relevant period",
        "Scaler was fitted only on earlier training data",
        "Later data was not used to fit the scaler or K-Means model"
    ]
})

leakage_audit

,Feature_or_Process,Leakage_Risk,Audit_Result
0,content_hash_id,Low,Used only as an identifier and excluded from K...
1,Search impressions,Low,Observed performance signal available in the r...
2,Search clicks,Low,Observed performance signal available in the r...
3,Average search position,Low,Observed performance signal available in the r...
4,Pageviews and sessions,Low,Observed performance signals available in the ...
5,Traffic-source sessions,Low,Observed performance signals available in the ...
6,Engagement and scroll events,Low,Observed performance signals available in the ...
7,Feature scaling,Controlled,Scaler was fitted only on earlier training data
8,Time-aware evaluation,Controlled,Later data was not used to fit the scaler or K...


## 3. Leakage Audit

I checked whether future information or identifiers were accidentally used as model features.

`content_hash_id` was used only to aggregate observations and was excluded from the clustering features.

The time-aware validation design separates earlier observations from later observations. The StandardScaler was fitted only on earlier data, and the K-Means model was also fitted only on earlier data.

Later-period data was used only for evaluation. This reduces the risk of future information leaking into model fitting.

However, this audit does not prove that all forms of leakage are impossible. The results should be interpreted as an observed validation check rather than a guarantee of production robustness.

In [ ]:
# Calculate distance from each evaluation content item
# to every cluster center

distances = kmeans.transform(X_test_scaled)

# Sort distances for each content item
sorted_distances = np.sort(distances, axis=1)

# Small margin = less clear cluster assignment
test_margin = (
    sorted_distances[:, 1] -
    sorted_distances[:, 0]
)

failure_examples = test_content[
    ["content_hash_id"] + features
].copy()

failure_examples["predicted_cluster"] = test_labels
failure_examples["assignment_margin"] = test_margin

# Lowest margins are the most ambiguous examples
failure_examples = failure_examples.sort_values(
    "assignment_margin",
    ascending=True
)

print("Most ambiguous cluster assignments:")
failure_examples[
    [
        "content_hash_id",
        "predicted_cluster",
        "assignment_margin",
        "total_impressions",
        "total_clicks",
        "total_pageviews",
        "organic_sessions"
    ]
].head(10)

Most ambiguous cluster assignments:


,content_hash_id,predicted_cluster,assignment_margin,total_impressions,total_clicks,total_pageviews,organic_sessions
8476,content_ae63a25825045384,0,0.000120,114,0,0,0
1092,content_179a20261f5ff964,0,0.000161,170,0,0,0
7677,content_9d8204e19b49515e,2,0.000168,2537,11,0,0
3466,content_47570b9e12bc638e,1,0.000488,24,0,0,0
1806,content_25c1ba9b57a10cc8,1,0.000639,683,1,0,0
2804,content_39d1f762cea9da37,0,0.000855,621,3,0,0
3067,content_3ec25c378d3bc745,0,0.001046,2,0,0,0
2683,content_378d91b9d3058837,0,0.001190,8,0,0,0
5573,content_72bc900c88bb4fa5,5,0.001537,558,0,0,0
8203,content_a8c31778929dc8a5,5,0.001623,249,0,0,0


## 4. Real Failure Examples and Interpretation

The examples with the smallest assignment margins are the most ambiguous cases because their distance to competing cluster centers is relatively similar.

These cases show that clustering does not create perfectly separated groups. Some content items have mixed performance characteristics and may not fit cleanly into a single archetype.

For example, a content item could have relatively high impressions but low clicks, or reasonable traffic but weak engagement. Such combinations can place an item near a cluster boundary.

Therefore, cluster assignments should be treated as decision-support signals rather than definitive labels.

## 4. Claim Rewrite

### Claim 1

**Original claim:**

The model identifies successful and unsuccessful content.

**Rewritten claim:**

The model identified clusters with different observed patterns of search visibility, traffic, and engagement. These clusters are descriptive content-performance archetypes and do not represent independently verified labels of success or failure.

---

### Claim 2

**Original claim:**

Higher impressions and clicks cause better content performance.

**Rewritten claim:**

The higher-performing cluster showed higher observed impressions and clicks. This clustering analysis identifies associations in the selected data and does not establish that these metrics cause better content performance.

---

### Claim 3

**Original claim:**

The model will work reliably on future content.

**Rewritten claim:**

The time-aware evaluation measured how the clustering structure behaved on a later period of the available data. This provides evidence about observed stability, but it does not guarantee future performance.

---

### Intended Use

The model is intended as a decision-support tool for grouping content into broad observed performance patterns and prioritizing further investigation.

The model should not be used as an automatic decision system, a causal model, or proof that a specific content item is inherently successful or unsuccessful.

In [ ]:
print("=" * 60)
print("WEEK 06 VALIDATION AUDIT — SELF-CHECK")
print("=" * 60)

checks = {
    "Two research-paper findings reviewed": True,
    "Constructive methodology question for Finding 1": True,
    "Constructive methodology question for Finding 2": True,
    "Random/previous validation result included": True,
    "Time-aware split created": True,
    "Earlier data used for model fitting": True,
    "Later data used for evaluation": True,
    "Before/after comparison created": True,
    "Leakage audit completed": True,
    "content_hash_id excluded from model features": True,
    "Scaler fitted only on training data": True,
    "Real ambiguous/failure examples examined": True,
    "Strong claims rewritten using safe language": True,
    "Causation not claimed": True,
    "Model described as decision-support": True
}

for check, status in checks.items():
    symbol = "✓" if status else "✗"
    print(f"{symbol} {check}")

print("\nMODEL RESULTS")
print("-" * 60)

print(f"Best time-aware K: {best_time_k}")
print(f"Time-aware silhouette score: {time_aware_score:.4f}")

print("\nBEFORE VS AFTER VALIDATION")
print("-" * 60)
print(comparison_df.to_string(index=False))

print("\nCLAIM STANDARD")
print("-" * 60)
print("Results describe observed patterns.")
print("Results do not establish causation.")
print("Model outputs are intended for decision-support.")
print("Future performance is not guaranteed.")

print("\n" + "=" * 60)
print("WEEK 06 VALIDATION AUDIT COMPLETE")
print("=" * 60)

WEEK 06 VALIDATION AUDIT — SELF-CHECK
✓ Two research-paper findings reviewed
✓ Constructive methodology question for Finding 1
✓ Constructive methodology question for Finding 2
✓ Random/previous validation result included
✓ Time-aware split created
✓ Earlier data used for model fitting
✓ Later data used for evaluation
✓ Before/after comparison created
✓ Leakage audit completed
✓ content_hash_id excluded from model features
✓ Scaler fitted only on training data
✓ Real ambiguous/failure examples examined
✓ Strong claims rewritten using safe language
✓ Causation not claimed
✓ Model described as decision-support

MODEL RESULTS
------------------------------------------------------------
Best time-aware K: 2
Time-aware silhouette score: 0.8808

BEFORE VS AFTER VALIDATION
------------------------------------------------------------
        Validation Design  Best K           Evaluation Metric    Score
Week 5 Random 80/20 Split       2   Held-out silhouette score 0.000000
  Week 6 Time-Aware 